# Multilingual Verifier Fine-Tuning
This notebook fine-tunes a multilingual HuggingFace encoder model for verifier scoring using multilingual trace-level data.

In [ ]:
import importlib

try:
    drive = importlib.import_module("google.colab").drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print("Google Colab is not available here; skipping drive mount.")

In [ ]:
%pip install -q transformers evaluate tomli sentencepiece accelerate

In [ ]:
import os
import re
import json
import time
import random
import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split


import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)

from sklearn.metrics import accuracy_score
import evaluate


In [ ]:
torch.set_default_dtype(torch.float32)


In [ ]:
f1_metric = evaluate.load("f1")

def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))


def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[?\s*Label\s*\]?:\s*(True|False|Conflicting|Unknown|Supports|Refutes))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")


def print_trainable_parameters(model):
    trainable_params, all_param = 0, 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_param} || "
        f"trainable%: {100 * trainable_params / all_param:.2f}"
    )


In [ ]:
class CustomClassifier(torch.nn.Module):
    def __init__(
        self,
        model_name,
        num_labels=1,
        freeze_base_layer=False,
    ):
        super().__init__()

        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            ignore_mismatched_sizes=True #
        )

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.logits


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
sep = tokenizer.sep_token

def stringify_evidences(evidences):
    if isinstance(evidences, list): return ' '.join(str(x).strip() for x in evidences[:3])
    return str(evidences)

def build_trace_level_dataframe(samples):
    rows = []
    for sample in samples:
        claim = str(sample.get('claim', '')).strip()
        gold_label = str(sample.get('label', sample.get('Verdict', ''))).strip().lower()
        evidence_text = stringify_evidences(sample.get('evidences', sample.get('Questions', '')))
        
        for verdict, trace in zip(sample.get('Verdict_list', []), sample.get('Reasoning_traces', [])):
            justification = trace.split('Label:')[0].strip()
            class_label = int(str(verdict).lower() == gold_label)
            input_text = f'Claim: {claim} {sep} Evidence: {evidence_text} {sep} Justification: {justification}'
            rows.append({'input_text': input_text, 'Class': class_label})
    return pd.DataFrame(rows)

# Încărcare și Split (Anti-Leakage)
def load_raw(path): return [json.loads(l) for l in open(path, 'r', encoding='utf-8') if l.strip()]
all_raw = load_raw(TRAIN_CSV) + load_raw(TRAIN_ARABIC_JSONL) + load_raw(TRAIN_SPANISH_JSONL)

train_raw, val_raw = train_test_split(all_raw, test_size=0.15, random_state=42)
train_df = build_trace_level_dataframe(train_raw).dropna().reset_index(drop=True)
val_df = build_trace_level_dataframe(val_raw).dropna().reset_index(drop=True)

In [ ]:
class TrainerModule:
    def __init__(self, model, train_loader, val_loader, tokenizer):
        self.device = torch.device('cuda')
        self.model = model.to(self.device)
        self.train_loader, self.val_loader = train_loader, val_loader
        self.tokenizer = tokenizer
        self.optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
        
        # FIX PENTRU BIAS: Greutate pentru clasa pozitivă (True)
        self.loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2.5]).to(self.device))
        self.scaler = torch.amp.GradScaler('cuda') if USE_AMP else None

    def train(self):
        for epoch in range(EPOCHS):
            self.model.train()
            for step, batch in enumerate(tqdm(self.train_loader)):
                ids, mask, labels = batch['input_ids'].to(self.device), batch['attention_mask'].to(self.device), batch['labels'].to(self.device)
                with torch.amp.autocast('cuda', enabled=USE_AMP):
                    logits = self.model(ids, mask)
                    loss = self.loss_fn(logits.squeeze(1), labels) / GRAD_ACCUM_STEPS
                self.scaler.scale(loss).backward()
                if (step + 1) % GRAD_ACCUM_STEPS == 0:
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.optimizer.zero_grad()
            self.evaluate()
            
    def evaluate(self):
        self.model.eval()
        preds, labels_all = [], []
        with torch.no_grad():
            for b in self.val_loader:
                logits = self.model(b['input_ids'].to(self.device), b['attention_mask'].to(self.device))
                preds.extend((torch.sigmoid(logits.squeeze(1)) >= 0.5).cpu().numpy().astype(int))
                labels_all.extend(b['labels'].numpy().astype(int))
        print(f'Val F1: {f1_metric.compute(predictions=preds, references=labels_all)["f1"]:.4f}')

In [ ]:
import os, json, re, datetime, torch, warnings
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, get_cosine_schedule_with_warmup
import evaluate

warnings.filterwarnings('ignore')
f1_metric = evaluate.load('f1')

# ===== CONFIGURAȚII OPTIMIZATE =====
BASE_MODEL = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'
MAX_LENGTH = 384
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 16
USE_AMP = True
EPOCHS = 4
LR = 2e-5
OUTPUT_DIR = './output_models'
TRAIN_CSV = '../English/English_train.jsonl' # Ajustează căile dacă e necesar
TRAIN_ARABIC_JSONL = '../Arabic/arabic_train.jsonl'
TRAIN_SPANISH_JSONL = '../spanish/spanish_train.jsonl'

In [ ]:
class VerifierEvaluator:
    def __init__(
        self,
        model_path,
        tokenizer_path,
        base_model,
        use_decomp,
        device="cuda",
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.use_decomp = use_decomp

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = CustomClassifier(
            base_model,
        )
        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

    def _stringify_evidence(self, evidences):
        if evidences is None:
            return ""
        if isinstance(evidences, str):
            return evidences
        if isinstance(evidences, list):
            return " ".join(str(x).strip() for x in evidences[:3] if str(x).strip())
        return str(evidences)

    def encode_input(self, claim, evidences, verdict, justification, max_length=None):
        evidence_text = self._stringify_evidence(evidences)
        if max_length is None:
            max_length = MAX_LENGTH if "MAX_LENGTH" in globals() else 384

        text = (
            f"Claim: {claim}\n"
            f"Verdict: {verdict}\n"
            f"Evidence: {evidence_text}\n"
            f"Justification: {justification}"
        )

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, evidences, verdict, justification):
        ids, mask = self.encode_input(claim, evidences, verdict, justification)
        with torch.no_grad():
            return float(self.model(ids, mask).item())

In [ ]:
# Sanity check jsonl inputs before training
required_cols = {"input_text", "Class"}

for name, path in [
    ("English", TRAIN_CSV),
    ("Arabic", TRAIN_ARABIC_JSONL),
    ("Spanish", TRAIN_SPANISH_JSONL),
]:
    df = pd.read_json(path, lines=True)
    missing = required_cols.difference(df.columns)
    if missing:
        raise ValueError(f"{name} jsonl missing columns: {sorted(missing)}")
    print(f"{name} rows: {len(df)}")

In [ ]:
from huggingface_hub import login


hf_token = "token"
if hf_token:
    login(token=hf_token)
else:
    print("HF_TOKEN is not set. Skipping login; this notebook assumes you are already authenticated.")


def load_raw_samples(path, lang):
    with open(path, "r", encoding="utf-8") as f:
        samples = [json.loads(line) for line in f if line.strip()]
    for sample in samples:
        sample["lang"] = lang
    return samples


def build_trace_level_dataframe(raw_samples):
    dataframe = pd.DataFrame(raw_samples).copy()
    required_cols = {"input_text", "Class"}
    missing = required_cols.difference(dataframe.columns)
    if missing:
        raise ValueError(f"Raw samples are missing columns: {sorted(missing)}")
    dataframe = dataframe.dropna(subset=["input_text", "Class"]).reset_index(drop=True)
    dataframe["Class"] = dataframe["Class"].astype(int)
    return dataframe


english_raw = load_raw_samples(TRAIN_CSV, "en")
arabic_raw = load_raw_samples(TRAIN_ARABIC_JSONL, "ar")
spanish_raw = load_raw_samples(TRAIN_SPANISH_JSONL, "es")

all_raw_samples = english_raw + arabic_raw + spanish_raw
raw_labels = [sample["Class"] for sample in all_raw_samples]

train_raw, val_raw = train_test_split(
    all_raw_samples,
    test_size=0.15,
    stratify=raw_labels,
    random_state=42,
)

train_df = build_trace_level_dataframe(train_raw)
val_df = build_trace_level_dataframe(val_raw)

if "TextDataset" not in globals():
    class TextDataset(Dataset):
        def __init__(self, dataframe, tokenizer, max_length):
            self.texts = dataframe["input_text"].tolist()
            self.labels = dataframe["Class"].tolist()
            self.tokenizer = tokenizer
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            self.max_length = max_length

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            encoding = self.tokenizer(
                self.texts[idx],
                truncation=True,
                padding="max_length",
                max_length=self.max_length,
                return_tensors="pt",
            )
            item = {k: v.squeeze(0) for k, v in encoding.items()}
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
            return item


tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = TextDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = TextDataset(val_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, pin_memory=True)

model = CustomClassifier(
    BASE_MODEL,
    use_lora=False,
    is_base_encoder=True,
)

print_trainable_parameters(model)

trainer = TrainerModule(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LR,
    patience=2,
    output_dir=OUTPUT_DIR,
    tokenizer=tokenizer,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    use_amp=USE_AMP,
)

trainer.train()

In [ ]:
# Write atomically to avoid partial/overwritten outputs if interrupted.
tmp_path = PREDICTIONS_JSON + ".tmp"
with open(tmp_path, "w", encoding="utf-8") as fp:
    json.dump(predictions, fp, indent=4, ensure_ascii=False)
os.replace(tmp_path, PREDICTIONS_JSON)
print(f"Saved {len(predictions)} claim-level predictions to {PREDICTIONS_JSON}")

In [ ]:
with open(PREDICTIONS_JSON, "w", encoding="utf-8") as fp:
    json.dump(predictions, fp, indent=4, ensure_ascii=False)

print(f"Saved {len(predictions)} claim-level predictions to {PREDICTIONS_JSON}")


In [ ]:
# Run predictions on VAL_CLAIMS_JSON (final test) and save atomically
import json
import os
import numpy as np

with open(VAL_CLAIMS_JSON, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Running predictions on {len(test_data)} samples from {VAL_CLAIMS_JSON}")

evaluator = VerifierEvaluator(
    model_path=f"{BEST_CHECKPOINT_DIR}/pytorch_model.bin",
    tokenizer_path=BEST_CHECKPOINT_DIR,
    base_model=BASE_MODEL,
    use_decomp=False
)

predictions = []

def capitalize_verdict(v):
    v_lower = str(v).lower().strip()
    if v_lower == "false":
        return "False"
    if v_lower == "true":
        return "True"
    if v_lower == "conflicting":
        return "Conflicting"
    return v_lower.capitalize()

for idx, sample in enumerate(test_data):
    verdict_list = []
    verifier_score_list = []
    justification_list = []

    for trace_idx, raw_trace in enumerate(sample.get("Reasoning_traces", [])):
        justification = remove_label_pattern(raw_trace).split("Label:")[0].strip()
        # Some test entries might not have Verdict_list aligned; guard access
        vlist = sample.get("Verdict_list", [])
        verdict = capitalize_verdict(vlist[trace_idx]) if trace_idx < len(vlist) else "Conflicting"

        score = evaluator.score(
            sample.get("claim", ""),
            sample.get("evidences", ""),
            verdict,
            justification,
        )

        verdict_list.append(verdict)
        justification_list.append(justification)
        verifier_score_list.append(score)

    if len(verifier_score_list) == 0:
        # No traces: default to Conflicting
        best_verdict = "Conflicting"
    else:
        ranked_indices = np.argsort(np.array(verifier_score_list))[::-1]
        best_verdict = verdict_list[int(ranked_indices[0])]

    predictions.append({
        "query_id": sample.get("query_id", idx),
        "Claim": sample.get("claim", ""),
        "label": best_verdict,
        "Verdict_BoN": best_verdict,
        "BoN_Verdict_list": verdict_list,
        "Reasoning_traces": justification_list,
        "score_list": verifier_score_list,
    })

# Atomic write
tmp_path = PREDICTIONS_JSON + ".tmp"
with open(tmp_path, "w", encoding="utf-8") as fp:
    json.dump(predictions, fp, indent=4, ensure_ascii=False)
os.replace(tmp_path, PREDICTIONS_JSON)
print(f"Saved {len(predictions)} claim-level predictions to {PREDICTIONS_JSON}")